In [17]:
# Importaciones
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn
from scipy.stats import poisson,skellam

In [18]:
# Data reading
def read_data(path):
    """To read a csv file and return a pandas df"""
    df = pd.read_csv(path)
    return df

In [19]:
from dotenv import load_dotenv
import os

load_dotenv()

sp_2526_path = os.getenv("SP_2025_2026")
sp_2627_path = os.getenv("SP_2026_2027")

In [20]:
sp1 = read_data(sp_2526_path)
sp2 = pd.read_csv(sp_2526_path)


In [21]:
# concatenar
sp_2627 = pd.concat([sp1,sp2], ignore_index=True)

sp_2627 = sp_2627[['HomeTeam','AwayTeam','FTHG','FTAG']]
sp_2627 = sp_2627.rename(columns={'FTHG': 'HomeGoals', 'FTAG': 'AwayGoals'})
#sp_2627.head()
sp_2627

,HomeTeam,AwayTeam,HomeGoals,AwayGoals
0,Girona,Vallecano,1,3
1,Villarreal,Oviedo,2,0
2,Mallorca,Barcelona,0,3
3,Alaves,Levante,2,1
4,Valencia,Sociedad,1,1
...,...,...,...,...
755,Girona,Elche,1,1
756,Mallorca,Oviedo,3,0
757,Real Madrid,Ath Bilbao,4,2
758,Valencia,Barcelona,3,1


In [22]:
sp_2627 = sp_2627[:-10]
sp_2627.mean(numeric_only=True)

HomeGoals    1.565333
AwayGoals    1.124000
dtype: float64

In [23]:
# probability of draw between home and away team
skellam.pmf(0.0,  sp_2627.mean(numeric_only=True).iloc[0],  sp_2627.mean(numeric_only=True).iloc[1])

np.float64(0.2514965552904865)

In [24]:
# probability of home team winning by one goal
skellam.pmf(1,  sp_2627.mean(numeric_only=True).iloc[0],  sp_2627.mean(numeric_only=True).iloc[1])

np.float64(0.23169101282568436)

In [25]:
# importing the tools required for the Poisson regression model
import statsmodels.api as sm
import statsmodels.formula.api as smf

goal_model_data = pd.concat([sp_2627[['HomeTeam','AwayTeam','HomeGoals']].assign(home=1).rename(
            columns={'HomeTeam':'team', 'AwayTeam':'opponent','HomeGoals':'goals'}),
           sp_2627[['AwayTeam','HomeTeam','AwayGoals']].assign(home=0).rename(
            columns={'AwayTeam':'team', 'HomeTeam':'opponent','AwayGoals':'goals'})])

poisson_model = smf.glm(formula="goals ~ home + team + opponent", data=goal_model_data,
                        family=sm.families.Poisson()).fit()
poisson_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                  goals   No. Observations:                 1500
Model:                            GLM   Df Residuals:                     1460
Model Family:                 Poisson   Df Model:                           39
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2033.7
Date:                Fri, 11 Sep 2026   Deviance:                       1324.7
Time:                        14:30:35   Pearson chi2:                 1.14e+03
No. Iterations:                     5   Pseudo R-squ. (CS):             0.1702
Covariance Type:            nonrobust                                         
===========================================================================================
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                   0.0493      0.148      0.334      0.739      -0.240       0.339
team[T.Ath Bilbao]         -0.0389      0.153     -0.254      0.799      -0.339       0.261
team[T.Ath Madrid]          0.3306      0.140      2.356      0.018       0.056       0.606
team[T.Barcelona]           0.7555      0.130      5.822      0.000       0.501       1.010
team[T.Betis]               0.2859      0.142      2.013      0.044       0.008       0.564
team[T.Celta]               0.1863      0.145      1.283      0.199      -0.098       0.471
team[T.Elche]               0.1095      0.148      0.740      0.459      -0.180       0.399
team[T.Espanol]            -0.0180      0.153     -0.118      0.906      -0.317       0.281
team[T.Getafe]             -0.3391      0.166     -2.048      0.041      -0.664      -0.015
team[T.Girona]             -0.1187      0.157     -0.758      0.449      -0.426       0.188
team[T.Levante]             0.0700      0.149      0.469      0.639      -0.223       0.363
team[T.Mallorca]            0.0514      0.150      0.342      0.732      -0.243       0.346
team[T.Osasuna]             0.0007      0.151      0.004      0.996      -0.296       0.297
team[T.Oviedo]             -0.5126      0.175     -2.921      0.003      -0.857      -0.169
team[T.Real Madrid]         0.5280      0.135      3.912      0.000       0.263       0.793
team[T.Sevilla]             0.0586      0.150      0.391      0.696      -0.235       0.352
team[T.Sociedad]            0.3021      0.142      2.130      0.033       0.024       0.580
team[T.Valencia]            0.0208      0.151      0.138      0.890      -0.275       0.317
team[T.Vallecano]          -0.0971      0.155     -0.626      0.531      -0.401       0.207
team[T.Villarreal]          0.4599      0.137      3.359      0.001       0.192       0.728
opponent[T.Ath Bilbao]      0.0302      0.134      0.225      0.822      -0.233       0.294
opponent[T.Ath Madrid]     -0.2508      0.146     -1.723      0.085      -0.536       0.035
opponent[T.Barcelona]      -0.4082      0.154     -2.653      0.008      -0.710      -0.107
opponent[T.Betis]          -0.1298      0.140     -0.925      0.355      -0.405       0.145
opponent[T.Celta]          -0.1252      0.140     -0.895      0.371      -0.399       0.149
opponent[T.Elche]           0.0354      0.134      0.264      0.792      -0.228       0.298
opponent[T.Espanol]        -0.0057      0.135     -0.042      0.966      -0.271       0.260
opponent[T.Getafe]         -0.3819      0.149     -2.558      0.011      -0.675      -0.089
opponent[T.Girona]         -0.0123      0.135     -0.091      0.928      -0.277       0.253
opponent[T.Levante]         0.0995      0.132      0.752      0.452      -0.160       0.359
opponent[T.Mallorca]        0.0352      0.134  

In [26]:
poisson_model.predict(pd.DataFrame(data={'team': 'Barcelona', 'opponent': 'Levante',
                                       'home':1},index=[1]))

1    3.44217
dtype: float64

In [27]:
poisson_model.predict(pd.DataFrame(data={'team': 'Levante', 'opponent': 'Barcelona',
                                       'home':0},index=[1]))

1    0.74912
dtype: float64

In [28]:
def simulate_match(foot_model, homeTeam, awayTeam, max_goals=10):
    home_goals_avg = foot_model.predict(pd.DataFrame(data={'team': homeTeam, 
                                                            'opponent': awayTeam,'home':1},
                                                      index=[1])).values[0]
    away_goals_avg = foot_model.predict(pd.DataFrame(data={'team': awayTeam, 
                                                            'opponent': homeTeam,'home':0},
                                                      index=[1])).values[0]
    team_pred = [[poisson.pmf(i, team_avg) for i in range(0, max_goals+1)] for team_avg in [home_goals_avg, away_goals_avg]]
    return(np.outer(np.array(team_pred[0]), np.array(team_pred[1])))
simulate_match(poisson_model, 'Barcelona', 'Levante', max_goals=3)

array([[0.01512676, 0.01133176, 0.00424442, 0.00105986],
       [0.05206889, 0.03900583, 0.01461002, 0.00364822],
       [0.08961498, 0.06713235, 0.02514508, 0.00627889],
       [0.10282334, 0.07702699, 0.02885122, 0.00720434]])

In [29]:
fcb_levante = simulate_match(poisson_model, "Barcelona", "Levante", max_goals=10)
# FCB win
np.sum(np.tril(fcb_levante, -1))

np.float64(0.86595680980541)

In [30]:
# Draw
np.sum(np.diag(fcb_levante))

np.float64(0.08777189048633198)

In [31]:
# levante win
np.sum(np.triu(fcb_levante, 1))

np.float64(0.04537771994831023)